<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "478"
iso_country = "Mauritania"
year = 2020
import_portfolio = "h5_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h5_ag6_by_sea_X.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 8.260*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Mauritania': 

Number of trade records present in dataset: 5975
Number of partner countries present in dataset: 115
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$2395.5m
Total weight within the dataset: 3496.8 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,478,Mauritania,156,China,Import,Sea,551329,"551329 | Fabrics, woven; dyed",5513,5513 | Woven fabrics of syn,55,55 | Man-made staple fibr,2676,"Woven fabrics of man-made staple fibres, conta...",Textiles and Wearing Apparel,27,TEX - Textiles,37766.991,NaN,37766.991,kg,89671.0,89671.0,5819.893313,19544.0,12144.074824,70677.219862,['Container'],['Container'],314.930862,314.174236,314.238616,314.238616,314.651908,314.651908,329.919239,596.889260,364.281278,337.079734,338.123554,347.083833,322.822614,578.734919,382.023297,350.309123,342.287204,358.363541,316.600349,632.921473,475.539491,415.221856,384.028054,373.847983,310.483311,672.541514,548.520306,513.597173,410.094453,401.152138,304.480815,683.343554,649.383642,656.693593,444.353042,421.592121,298.359912,669.743819,642.626988,696.391169,517.070004,429.406429,292.584988,596.296217,618.360923,648.302704,485.360389,434.540791,286.714258,543.755431,554.247558,675.920775,557.658140,437.592699,2676
1,1,478,Mauritania,156,China,Import,Sea,551511,"551511 | Fabrics, woven; of p",5515,5515 | Woven fabrics of syn,55,55 | Man-made staple fibr,2679,Other woven fabrics of man-made staple fibres,Textiles and Wearing Apparel,27,TEX - Textiles,7404.441,NaN,7404.441,kg,11705.0,11705.0,875.204926,19544.0,12144.074824,10628.554110,['Container'],['Container'],47.359810,47.246027,47.255709,47.255709,47.317860,47.317860,49.613786,89.761168,54.781205,50.690593,50.847564,52.195026,48.546584,87.031089,57.449278,52.680050,51.473701,53.891286,47.610870,95.179750,71.512394,62.441731,57.750757,56.219862,46.690980,101.137876,82.487367,77.235570,61.670664,60.325904,45.788315,102.762304,97.655358,98.754640,66.822526,63.399702,44.867844,100.717154,96.639281,104.724425,77.757820,64.574830,43.999402,89.671985,92.990111,97.492804,72.989277,65.346944,43.116552,81.770817,83.348640,101.646054,83.861529,65.805895,2679


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $728.7m (30.4%), ... 

                              cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                             
10 | Cereals               201510724.2  683879502.0      8.4     19.6
17 | Sugars and sugar con  165107958.0  360988576.7      6.9     10.3
15 | Animal, vegetable or  150995789.0  153070959.1      6.3      4.4
04 | Dairy produce; birds   75274303.8   43824058.5      3.1      1.3
07 | Vegetables and certa   45218729.9   89315193.5      1.9      2.6
02 | Meat and edible meat   23942461.3   12885126.4      1.0      0.4
09 | Coffee, tea, mate an   20055692.3    5284763.6      0.8      0.2
19 | Preparations of cere   15944543.4    5956015.5      0.7      0.2
21 | Miscellaneous edible    7152472.6    2579967.3      0.3      0.1
08 | Fruit and nuts, edib    6538734.0    4619243.3      0.3      0.1
20 | Preparations of vege    5749568.4    4587592.1      0.

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $570.0m (23.8%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Agriculture       350451897.9  358751561.1     14.6     10.3
Food & Beverages  219359129.8  402854265.8      9.2     11.5
Fishing              238366.9      47825.6      0.0      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $546.1m (22.8%), ... 

                                  cifvalue   vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                                
SGR - Sugar                    160086087.4  351396902.8      6.7     10.0
VOL - Vegetable oils and fats  153214738.4  159777071.6      6.4      4.6
MIL - Dairy products            75124989.3   43666255.5      3.1      1.2
V_F - Vegetables, fruit, nuts   51438554.4   93698561.4      2.1      2.7
OFD - Food products n.e.c.      48829062.2   32728863.2      2.0      0.9
PCR - Processed rice            24962347.3   43359597.3      1.0      1.2
OMT - Meat products n.e.c.      23401874.0   12791691.3      1.0      0.4
OCR - Crops n.e.c.               7409239.5    1967362.3      0.3      0.1
GRO - Cereal grains n.e.c.        692098.1    2862863.8      0.0      0.1
CMT - Bovine meat prods           532363.1      86616.3      0.0      0.0
PDR - Paddy rice                  277692.8    

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 30.4% 
GTAP 22.8% 
EORA 23.8%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46394/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,478,Mauritania,156,China,Import,Sea,551329,"551329 | Fabrics, woven; dyed",5513,5513 | Woven fabrics of syn,55,55 | Man-made staple fibr,2676,"Woven fabrics of man-made staple fibres, conta...",Textiles and Wearing Apparel,27,TEX - Textiles,37766.991,NaN,37766.991,kg,89671.0,89671.0,5819.893313,19544.0,12144.074824,70677.219862,['Container'],['Container'],314.930862,314.174236,314.238616,314.238616,314.651908,314.651908,329.919239,596.889260,364.281278,337.079734,338.123554,347.083833,322.822614,578.734919,382.023297,350.309123,342.287204,358.363541,316.600349,632.921473,475.539491,415.221856,384.028054,373.847983,310.483311,672.541514,548.520306,513.597173,410.094453,401.152138,304.480815,683.343554,649.383642,656.693593,444.353042,421.592121,298.359912,669.743819,642.626988,696.391169,517.070004,429.406429,292.584988,596.296217,618.360923,648.302704,485.360389,434.540791,286.714258,543.755431,554.247558,675.920775,557.658140,437.592699,2676,NaN
1,1,478,Mauritania,156,China,Import,Sea,551511,"551511 | Fabrics, woven; of p",5515,5515 | Woven fabrics of syn,55,55 | Man-made staple fibr,2679,Other woven fabrics of man-made staple fibres,Textiles and Wearing Apparel,27,TEX - Textiles,7404.441,NaN,7404.441,kg,11705.0,11705.0,875.204926,19544.0,12144.074824,10628.554110,['Container'],['Container'],47.359810,47.246027,47.255709,47.255709,47.317860,47.317860,49.613786,89.761168,54.781205,50.690593,50.847564,52.195026,48.546584,87.031089,57.449278,52.680050,51.473701,53.891286,47.610870,95.179750,71.512394,62.441731,57.750757,56.219862,46.690980,101.137876,82.487367,77.235570,61.670664,60.325904,45.788315,102.762304,97.655358,98.754640,66.822526,63.399702,44.867844,100.717154,96.639281,104.724425,77.757820,64.574830,43.999402,89.671985,92.990111,97.492804,72.989277,65.346944,43.116552,81.770817,83.348640,101.646054,83.861529,65.805895,2679,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $683.9m of $2395.5m (28.5%)
Volume: 1347.0 of 3496.8 million kilograms (38.5%)
Value-to-Weight: 0.51 from 0.69 US$ per kg (+-25.9%)
Average distance to market (volume-weighted): 4080.3 from 3838.0 nm (+6.3%)

            food_group  N_com     cifvalue   vol_kg_vtw  pct_val  pct_vol
0  Unprocessed Cereals     17  176270684.1  640037552.3      7.4     18.3
1      Sugar and Honey     36  165114965.5  360990585.8      6.9     10.3
2        Fats and Oils     36  140220300.5  140465818.6      5.9      4.0
3                Dairy    121   75028043.1   43709783.1      3.1      1.3
4    Processed Cereals    192   49667098.8   53322116.9      2.1      1.5
5           Vegetables     99   47238047.3   90993099.7      2.0      2.6
6        Meat and Prep     40   23829141.0   12861408.4      1.0      0.4
7               Fruits     33    6538109.6    4618964.2      0.3      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,69547235.5,1.256975e+08,76760806.4,71038200.7,71258294.5,73157829.8,64192749.0,1.440601e+08,1.368461e+08,1.383869e+08,9.363174e+07,8.883800e+07,60453117.5,1.145349e+08,1.168139e+08,1.422648e+08,1.174733e+08,9.220476e+07
1,Fats and Oils,570581318.4,9.134647e+08,603758252.2,567841664.7,569505693.5,591520844.0,536816470.4,1.015180e+09,9.953253e+08,1.044186e+09,7.245705e+08,6.879200e+08,513247687.4,8.577336e+08,8.741354e+08,1.007837e+09,8.294359e+08,7.080537e+08
2,Fruits,5729193.8,1.036093e+07,6324911.3,5853854.1,5871880.7,6027034.1,5287753.3,1.186236e+07,1.127098e+07,1.139751e+07,7.713888e+06,7.320165e+06,4979446.0,9.443991e+06,9.625453e+06,1.173386e+07,9.677000e+06,7.598018e+06
3,Meat and Prep,26908460.5,4.868274e+07,29711054.1,27492475.8,27577610.5,28308417.6,24833683.5,5.573401e+07,5.296422e+07,5.356042e+07,3.624177e+07,3.438537e+07,23384629.8,4.434910e+07,4.520485e+07,5.512860e+07,4.548302e+07,3.569039e+07
4,Processed Cereals,112188240.8,1.875283e+08,120273247.0,112705920.9,113092902.7,116853474.7,104872257.4,2.190334e+08,2.067579e+08,2.116572e+08,1.478789e+08,1.376397e+08,99764178.0,1.781588e+08,1.839067e+08,2.135576e+08,1.777700e+08,1.418147e+08
5,Sugar and Honey,386029664.8,6.510215e+08,415188031.5,388504330.5,389827284.6,402516629.5,360359658.6,7.586575e+08,7.167236e+08,7.326985e+08,5.100857e+08,4.757246e+08,342435841.4,6.154805e+08,6.344842e+08,7.408943e+08,6.161877e+08,4.905517e+08
6,Unprocessed Cereals,838324187.9,1.516493e+09,925591076.1,856493880.7,859146658.0,881924090.6,773702612.0,1.736201e+09,1.649899e+09,1.668505e+09,1.129058e+09,1.071190e+09,728570005.2,1.381593e+09,1.408282e+09,1.717301e+09,1.416854e+09,1.111831e+09
7,Vegetables,109372454.9,1.974968e+08,120678018.9,111718699.1,112062192.1,115035920.5,100971600.4,2.261670e+08,2.148805e+08,2.173574e+08,1.471742e+08,1.396366e+08,95103940.6,1.801365e+08,1.836224e+08,2.236606e+08,1.844796e+08,1.449207e+08


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.75,0.10,0.02,0.02,0.05,1.06,0.97,0.99,0.39,0.33,0.72,0.75,1.09,0.76,0.42
1,Fats and Oils,2.45,0.24,-0.02,-0.01,0.15,3.41,3.27,3.62,1.34,1.08,2.46,2.57,3.53,2.25,1.39
2,Fruits,0.71,0.09,0.02,0.02,0.05,1.01,0.92,0.93,0.37,0.31,0.68,0.71,1.03,0.72,0.40
3,Meat and Prep,0.91,0.12,0.02,0.03,0.06,1.30,1.18,1.21,0.48,0.40,0.88,0.92,1.33,0.93,0.52
4,Processed Cereals,1.52,0.16,0.01,0.02,0.09,2.30,2.05,2.15,0.87,0.66,1.58,1.69,2.29,1.57,0.85
5,Sugar and Honey,1.60,0.18,0.01,0.02,0.10,2.41,2.16,2.26,0.91,0.70,1.65,1.77,2.41,1.66,0.90
6,Unprocessed Cereals,3.85,0.50,0.10,0.12,0.25,5.46,4.97,5.08,2.02,1.69,3.70,3.86,5.61,3.90,2.17
7,Vegetables,1.87,0.24,0.05,0.06,0.12,2.65,2.41,2.46,0.98,0.82,1.80,1.87,2.72,1.89,1.05
